In [20]:
import os

os.listdir('/content')

['.config',
 'retail-orders-raw.csv',
 'retail-data-dictionary.csv',
 'sample_data']

In [21]:
import pandas as pd

# Load the raw retail dataset
df = pd.read_csv('/content/retail-orders-raw.csv')

# Load the data dictionary
dictionary = pd.read_csv('/content/retail-data-dictionary.csv')

# Show basic information
print("DATASET SHAPE:")
print(df.shape)

print("\nDATASET COLUMNS:")
print(df.columns.tolist())

print("\nDATASET INFO:")
df.info()

print("\nDATA DICTIONARY:")
display(dictionary)

DATASET SHAPE:
(12, 9)

DATASET COLUMNS:
['order_id', 'order_date', 'customer_segment', 'city', 'category', 'quantity', 'unit_price', 'discount_pct', 'payment_status']

DATASET INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          12 non-null     object 
 1   order_date        11 non-null     object 
 2   customer_segment  12 non-null     object 
 3   city              11 non-null     object 
 4   category          12 non-null     object 
 5   quantity          12 non-null     object 
 6   unit_price        12 non-null     int64  
 7   discount_pct      11 non-null     float64
 8   payment_status    12 non-null     object 
dtypes: float64(1), int64(1), object(7)
memory usage: 996.0+ bytes

DATA DICTIONARY:


,column,data_type,business_definition,quality_rule
0,order_id,string,Unique order identifier,Required and unique
1,order_date,date,Date the customer placed the order,Required ISO date between 2025-01-01 and today
2,customer_segment,category,Commercial customer segment,Student Fresher or Professional after normaliz...
3,city,string,Customer billing city,Required non-empty text
4,category,category,Product family,Learning Kit Course Access or Mentor Session
5,quantity,integer,Units purchased,Whole number greater than zero
6,unit_price,decimal,Price per unit before discount,Non-negative INR amount
7,discount_pct,decimal,Percentage discount applied,Between 0 and 100; missing means zero only aft...
8,payment_status,category,Latest order settlement state,Paid Pending Failed or Refunded after normaliz...


In [22]:
# STEP 3: DATA QUALITY PROFILING

# 1. Missing values
print("=== MISSING VALUES ===")
print(df.isnull().sum())


# 2. Duplicate order IDs
print("\n=== DUPLICATE ORDER IDs ===")
print("Duplicate order IDs:", df['order_id'].duplicated().sum())


# 3. Check order dates
print("\n=== ORDER DATE CHECK ===")
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

print("Invalid/missing dates:", df['order_date'].isnull().sum())
print("Earliest date:", df['order_date'].min())
print("Latest date:", df['order_date'].max())


# 4. Check quantity
print("\n=== QUANTITY CHECK ===")
quantity_numeric = pd.to_numeric(df['quantity'], errors='coerce')

print("Non-numeric quantity:", quantity_numeric.isnull().sum())
print("Quantity <= 0:", (quantity_numeric <= 0).sum())


# 5. Check unit price
print("\n=== UNIT PRICE CHECK ===")
print("Negative unit prices:", (df['unit_price'] < 0).sum())


# 6. Check discount
print("\n=== DISCOUNT CHECK ===")
print("Missing discounts:", df['discount_pct'].isnull().sum())
print("Discount below 0:", (df['discount_pct'] < 0).sum())
print("Discount above 100:", (df['discount_pct'] > 100).sum())


# 7. Check category values
print("\n=== CATEGORY VALUES ===")
print(df['category'].value_counts(dropna=False))


# 8. Check customer segments
print("\n=== CUSTOMER SEGMENTS ===")
print(df['customer_segment'].value_counts(dropna=False))


# 9. Check payment status
print("\n=== PAYMENT STATUS ===")
print(df['payment_status'].value_counts(dropna=False))


# 10. Check cities
print("\n=== CITIES ===")
print(df['city'].value_counts(dropna=False))

=== MISSING VALUES ===
order_id            0
order_date          1
customer_segment    0
city                1
category            0
quantity            0
unit_price          0
discount_pct        1
payment_status      0
dtype: int64

=== DUPLICATE ORDER IDs ===
Duplicate order IDs: 1

=== ORDER DATE CHECK ===
Invalid/missing dates: 3
Earliest date: 2026-01-03 00:00:00
Latest date: 2026-01-18 00:00:00

=== QUANTITY CHECK ===
Non-numeric quantity: 1
Quantity <= 0: 1

=== UNIT PRICE CHECK ===
Negative unit prices: 0

=== DISCOUNT CHECK ===
Missing discounts: 1
Discount below 0: 0
Discount above 100: 1

=== CATEGORY VALUES ===
category
Learning Kit      5
Course Access     4
Mentor Session    3
Name: count, dtype: int64

=== CUSTOMER SEGMENTS ===
customer_segment
Student         4
Professional    4
Fresher         3
student         1
Name: count, dtype: int64

=== PAYMENT STATUS ===
payment_status
Paid        7
Pending     2
paid        1
Failed      1
Refunded    1
Name: count, dtype: in

In [23]:
# STEP 4: KPI CALCULATIONS

# Make a working copy
kpi_df = df.copy()

# Convert fields to the correct types
kpi_df['quantity'] = pd.to_numeric(kpi_df['quantity'], errors='coerce')

# Normalize text values
kpi_df['customer_segment'] = kpi_df['customer_segment'].str.title()
kpi_df['payment_status'] = kpi_df['payment_status'].str.title()

# Treat missing discount as 0 for KPI calculation
kpi_df['discount_pct'] = kpi_df['discount_pct'].fillna(0)

# Keep only records that meet basic calculation rules
valid = (
    kpi_df['order_date'].notna() &
    kpi_df['quantity'].notna() &
    (kpi_df['quantity'] > 0) &
    (kpi_df['unit_price'] >= 0) &
    (kpi_df['discount_pct'] >= 0) &
    (kpi_df['discount_pct'] <= 100)
)

clean_df = kpi_df[valid].copy()

print("Total records:", len(kpi_df))
print("Records used for KPI calculation:", len(clean_df))
print("Records excluded:", len(kpi_df) - len(clean_df))


# Calculate sales
clean_df['gross_sales'] = clean_df['quantity'] * clean_df['unit_price']

clean_df['discount_amount'] = (
    clean_df['gross_sales'] *
    clean_df['discount_pct'] / 100
)

clean_df['net_sales'] = (
    clean_df['gross_sales'] -
    clean_df['discount_amount']
)


# KPI calculations
total_orders = clean_df['order_id'].nunique()

paid_orders = clean_df.loc[
    clean_df['payment_status'] == 'Paid',
    'order_id'
].nunique()

paid_df = clean_df[clean_df['payment_status'] == 'Paid']

paid_net_sales = paid_df['net_sales'].sum()

units_sold = paid_df['quantity'].sum()

aov = paid_net_sales / paid_orders if paid_orders else 0

paid_order_rate = (
    paid_orders / total_orders * 100
    if total_orders else 0
)

average_discount = clean_df['discount_pct'].mean()

refunded_orders = clean_df.loc[
    clean_df['payment_status'] == 'Refunded',
    'order_id'
].nunique()

refund_rate = (
    refunded_orders / total_orders * 100
    if total_orders else 0
)


# Display KPIs
kpis = pd.DataFrame({
    'KPI': [
        'Total Orders',
        'Paid Orders',
        'Paid Order Rate',
        'Net Sales',
        'Units Sold',
        'Average Order Value',
        'Average Discount %',
        'Refund Rate'
    ],
    'Value': [
        total_orders,
        paid_orders,
        round(paid_order_rate, 2),
        round(paid_net_sales, 2),
        units_sold,
        round(aov, 2),
        round(average_discount, 2),
        round(refund_rate, 2)
    ]
})

display(kpis)

Total records: 12
Records used for KPI calculation: 7
Records excluded: 5


,KPI,Value
0,Total Orders,6.00
1,Paid Orders,3.00
2,Paid Order Rate,50.00
3,Net Sales,6791.50
4,Units Sold,9.00
5,Average Order Value,2263.83
6,Average Discount %,5.00
7,Refund Rate,16.67


In [24]:
# STEP 5: CREATE KPI DICTIONARY

kpi_dictionary = pd.DataFrame([
    {
        "KPI Name": "Total Orders",
        "Business Definition": "Number of unique valid orders",
        "Formula": "COUNT DISTINCT(order_id)",
        "Grain": "Order",
        "Filters": "Valid records only",
        "Owner": "E-commerce Manager",
        "Refresh Cadence": "Daily"
    },
    {
        "KPI Name": "Paid Orders",
        "Business Definition": "Number of unique orders with successful payment",
        "Formula": "COUNT DISTINCT(order_id) where payment_status = Paid",
        "Grain": "Order",
        "Filters": "Payment status = Paid",
        "Owner": "E-commerce Manager",
        "Refresh Cadence": "Daily"
    },
    {
        "KPI Name": "Paid Order Rate",
        "Business Definition": "Percentage of valid orders that are paid",
        "Formula": "Paid Orders / Total Orders × 100",
        "Grain": "Order",
        "Filters": "Valid records only",
        "Owner": "E-commerce Manager",
        "Refresh Cadence": "Daily"
    },
    {
        "KPI Name": "Net Sales",
        "Business Definition": "Sales after applying discounts for paid orders",
        "Formula": "Quantity × Unit Price × (1 - Discount % / 100)",
        "Grain": "Order line",
        "Filters": "Payment status = Paid",
        "Owner": "Finance Manager",
        "Refresh Cadence": "Daily"
    },
    {
        "KPI Name": "Units Sold",
        "Business Definition": "Total quantity of products sold in paid orders",
        "Formula": "SUM(quantity)",
        "Grain": "Order line",
        "Filters": "Payment status = Paid",
        "Owner": "E-commerce Manager",
        "Refresh Cadence": "Daily"
    },
    {
        "KPI Name": "Average Order Value",
        "Business Definition": "Average net sales generated per paid order",
        "Formula": "Net Sales / Paid Orders",
        "Grain": "Order",
        "Filters": "Payment status = Paid",
        "Owner": "E-commerce Manager",
        "Refresh Cadence": "Daily"
    },
    {
        "KPI Name": "Average Discount %",
        "Business Definition": "Average percentage discount applied",
        "Formula": "AVERAGE(discount_pct)",
        "Grain": "Order line",
        "Filters": "Valid records only",
        "Owner": "Pricing Manager",
        "Refresh Cadence": "Daily"
    },
    {
        "KPI Name": "Refund Rate",
        "Business Definition": "Percentage of valid orders that were refunded",
        "Formula": "Refunded Orders / Total Orders × 100",
        "Grain": "Order",
        "Filters": "Valid records only",
        "Owner": "Finance Manager",
        "Refresh Cadence": "Daily"
    }
])

# Display
display(kpi_dictionary)

# Save to Excel
kpi_dictionary.to_excel(
    "/content/KPI_Dictionary.xlsx",
    index=False
)

print("KPI Dictionary created successfully!")

,KPI Name,Business Definition,Formula,Grain,Filters,Owner,Refresh Cadence
0,Total Orders,Number of unique valid orders,COUNT DISTINCT(order_id),Order,Valid records only,E-commerce Manager,Daily
1,Paid Orders,Number of unique orders with successful payment,COUNT DISTINCT(order_id) where payment_status ...,Order,Payment status = Paid,E-commerce Manager,Daily
2,Paid Order Rate,Percentage of valid orders that are paid,Paid Orders / Total Orders × 100,Order,Valid records only,E-commerce Manager,Daily
3,Net Sales,Sales after applying discounts for paid orders,Quantity × Unit Price × (1 - Discount % / 100),Order line,Payment status = Paid,Finance Manager,Daily
4,Units Sold,Total quantity of products sold in paid orders,SUM(quantity),Order line,Payment status = Paid,E-commerce Manager,Daily
5,Average Order Value,Average net sales generated per paid order,Net Sales / Paid Orders,Order,Payment status = Paid,E-commerce Manager,Daily
6,Average Discount %,Average percentage discount applied,AVERAGE(discount_pct),Order line,Valid records only,Pricing Manager,Daily
7,Refund Rate,Percentage of valid orders that were refunded,Refunded Orders / Total Orders × 100,Order,Valid records only,Finance Manager,Daily


KPI Dictionary created successfully!


In [25]:
# STEP 6: CREATE DATA QUALITY CONTRACT

dq_contract = pd.DataFrame([
    {
        "Rule ID": "DQ001",
        "Dimension": "Completeness",
        "Field": "order_date",
        "Quality Rule": "Order date must be present",
        "Threshold": "0% missing",
        "Failure Action": "Reject record and investigate source data",
        "Escalation": "Notify Data Steward",
        "Owner": "Data Engineering"
    },
    {
        "Rule ID": "DQ002",
        "Dimension": "Uniqueness",
        "Field": "order_id",
        "Quality Rule": "Order ID must be unique",
        "Threshold": "0 duplicates",
        "Failure Action": "Quarantine duplicate records",
        "Escalation": "Notify Data Engineering",
        "Owner": "Data Engineering"
    },
    {
        "Rule ID": "DQ003",
        "Dimension": "Validity",
        "Field": "order_date",
        "Quality Rule": "Order date must be a valid date between 2025-01-01 and today",
        "Threshold": "0 invalid dates",
        "Failure Action": "Reject invalid records",
        "Escalation": "Notify Data Steward",
        "Owner": "Data Engineering"
    },
    {
        "Rule ID": "DQ004",
        "Dimension": "Validity",
        "Field": "quantity",
        "Quality Rule": "Quantity must be a whole number greater than zero",
        "Threshold": "0 violations",
        "Failure Action": "Reject invalid records",
        "Escalation": "Notify Data Engineering",
        "Owner": "Data Engineering"
    },
    {
        "Rule ID": "DQ005",
        "Dimension": "Validity",
        "Field": "unit_price",
        "Quality Rule": "Unit price must be non-negative",
        "Threshold": "0 violations",
        "Failure Action": "Reject invalid records",
        "Escalation": "Notify Finance/Data Steward",
        "Owner": "Finance Manager"
    },
    {
        "Rule ID": "DQ006",
        "Dimension": "Validity",
        "Field": "discount_pct",
        "Quality Rule": "Discount must be between 0 and 100",
        "Threshold": "0 violations",
        "Failure Action": "Reject invalid records",
        "Escalation": "Notify Pricing Manager",
        "Owner": "Pricing Manager"
    },
    {
        "Rule ID": "DQ007",
        "Dimension": "Consistency",
        "Field": "customer_segment",
        "Quality Rule": "Values must be normalized to Student, Fresher, or Professional",
        "Threshold": "0 invalid values",
        "Failure Action": "Normalize casing; reject unknown values",
        "Escalation": "Notify Data Steward",
        "Owner": "Data Steward"
    },
    {
        "Rule ID": "DQ008",
        "Dimension": "Consistency",
        "Field": "payment_status",
        "Quality Rule": "Values must be Paid, Pending, Failed, or Refunded",
        "Threshold": "0 invalid values",
        "Failure Action": "Normalize casing; reject unknown values",
        "Escalation": "Notify Data Steward",
        "Owner": "Data Steward"
    },
    {
        "Rule ID": "DQ009",
        "Dimension": "Completeness",
        "Field": "city",
        "Quality Rule": "City must be present and non-empty",
        "Threshold": "0% missing",
        "Failure Action": "Reject or quarantine record",
        "Escalation": "Notify Data Steward",
        "Owner": "Data Engineering"
    }
])

display(dq_contract)

# Save the contract
dq_contract.to_excel(
    "/content/Data_Quality_Contract.xlsx",
    index=False
)

print("Data Quality Contract created successfully!")

,Rule ID,Dimension,Field,Quality Rule,Threshold,Failure Action,Escalation,Owner
0,DQ001,Completeness,order_date,Order date must be present,0% missing,Reject record and investigate source data,Notify Data Steward,Data Engineering
1,DQ002,Uniqueness,order_id,Order ID must be unique,0 duplicates,Quarantine duplicate records,Notify Data Engineering,Data Engineering
2,DQ003,Validity,order_date,Order date must be a valid date between 2025-0...,0 invalid dates,Reject invalid records,Notify Data Steward,Data Engineering
3,DQ004,Validity,quantity,Quantity must be a whole number greater than zero,0 violations,Reject invalid records,Notify Data Engineering,Data Engineering
4,DQ005,Validity,unit_price,Unit price must be non-negative,0 violations,Reject invalid records,Notify Finance/Data Steward,Finance Manager
5,DQ006,Validity,discount_pct,Discount must be between 0 and 100,0 violations,Reject invalid records,Notify Pricing Manager,Pricing Manager
6,DQ007,Consistency,customer_segment,"Values must be normalized to Student, Fresher,...",0 invalid values,Normalize casing; reject unknown values,Notify Data Steward,Data Steward
7,DQ008,Consistency,payment_status,"Values must be Paid, Pending, Failed, or Refunded",0 invalid values,Normalize casing; reject unknown values,Notify Data Steward,Data Steward
8,DQ009,Completeness,city,City must be present and non-empty,0% missing,Reject or quarantine record,Notify Data Steward,Data Engineering


Data Quality Contract created successfully!


In [26]:
# RETAIL KPI DICTIONARY & DATA QUALITY PROJECT

print("""
PROJECT: Retail KPI Dictionary & Data Quality Contract

Decision Owner:
Head of E-commerce / Retail Business Manager

Dataset:
Retail Orders Dataset

Deliverables:
1. KPI Dictionary
2. Data Profile
3. Data Quality Contract

Data Quality Summary:
- Total records: 12
- Records used for KPI calculation: 7
- Records excluded: 5

Key data-quality issues identified:
- Missing order date
- Missing city
- Missing discount
- Duplicate order ID
- Invalid order dates
- Invalid quantity
- Discount above 100%
- Inconsistent customer-segment capitalization
- Inconsistent payment-status capitalization
""")


PROJECT: Retail KPI Dictionary & Data Quality Contract

Decision Owner:
Head of E-commerce / Retail Business Manager

Dataset:
Retail Orders Dataset

Deliverables:
1. KPI Dictionary
2. Data Profile
3. Data Quality Contract

Data Quality Summary:
- Total records: 12
- Records used for KPI calculation: 7
- Records excluded: 5

Key data-quality issues identified:
- Missing order date
- Missing city
- Missing discount
- Duplicate order ID
- Invalid order dates
- Invalid quantity
- Discount above 100%
- Inconsistent customer-segment capitalization
- Inconsistent payment-status capitalization



In [27]:
# FINAL OUTPUT FILES

import os

print("Generated files:")

for file in [
    "KPI_Dictionary.xlsx",
    "Data_Quality_Contract.xlsx"
]:
    path = "/content/" + file
    print(file, "->", "Created" if os.path.exists(path) else "Missing")

Generated files:
KPI_Dictionary.xlsx -> Created
Data_Quality_Contract.xlsx -> Created


In [28]:
from google.colab import files

files.download('/content/KPI_Dictionary.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
import os

print("KPI Dictionary exists:", os.path.exists("KPI_Dictionary.xlsx"))
print("Data Quality Contract exists:", os.path.exists("Data_Quality_Contract.xlsx"))

KPI Dictionary exists: True
Data Quality Contract exists: True


In [32]:
from google.colab import files

files.download("KPI_Dictionary.xlsx")
files.download("Data_Quality_Contract.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>